In [0]:
from pyspark.sql.functions import *

In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df=spark.createDataFrame(data,columns)

In [0]:
# store data as delta format
df.write.format("delta").mode("overWrite").save("/Volumes/medallion/bronze/bronze_data")
     

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df1= spark.read.format("delta").load("/Volumes/medallion/bronze/bronze_data")
display(df1)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df1=df1.dropDuplicates()
df1.display()

order_id,customer_id,product,category,city,date,amount,quantity
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
df1=df1.fillna({"city":"unknown","amount":0})
df1.display()

order_id,customer_id,product,category,city,date,amount,quantity
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
df=df1.withColumn("amount",col("amount").cast("int"))

In [0]:
df1 = df1.filter(col("amount" )> 0)
df1.display()

order_id,customer_id,product,category,city,date,amount,quantity
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
df1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = false)
 |-- date: string (nullable = true)
 |-- amount: string (nullable = false)
 |-- quantity: long (nullable = true)



In [0]:
df1.show()


+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     107|       C005|Headphones|Accessories|  unknown|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
df1.write.format("delta").mode("overWrite").save("/Volumes/medallion/silver/silver_data")

In [0]:
df1=spark.read.format("delta").load("/Volumes/medallion/silver/silver_data")
df1.display()

order_id,customer_id,product,category,city,date,amount,quantity
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
df2=spark.read.format("delta").load("/Volumes/medallion/silver/silver_data")
df2.display()


order_id,customer_id,product,category,city,date,amount,quantity
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
df2.write.format("delta").mode("append").save("/Volumes/medallion/gold/gold_data")

In [0]:
final_df=spark.read.format("delta").load("/Volumes/medallion/gold/gold_data")

In [0]:
final_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### Aggregations

In [0]:
products=final_df.groupBy("product").agg(sum("amount").alias("total_sales"))
products.display()

product,total_sales
Tablet,84000.0
Mobile,66000.0
Watch,16000.0
Headphones,6000.0
Laptop,210000.0


In [0]:
category=final_df.groupBy("category").agg(sum("amount"))
category.display()

category,sum(amount)
Electronics,360000.0
Accessories,22000.0


In [0]:
city=final_df.groupBy("city").agg(sum("amount"))
city.display()

city,sum(amount)
Delhi,110000.0
Chennai,66000.0
Hyderabad,184000.0
Mumbai,16000.0
unknown,6000.0


###customer metrics

In [0]:
customers=final_df.groupBy("order_id").agg(count("quantity").alias("total_order"))
customers.display()

order_id,total_order
104,2
109,2
106,2
101,2
107,2
103,4
105,2


In [0]:
total_spent=final_df.groupBy("customer_id").agg(sum("amount"))
total_spent.display()

customer_id,sum(amount)
C005,6000.0
C004,16000.0
C003,110000.0
C001,184000.0
C002,36000.0
C007,30000.0


### Top Analysis

In [0]:
top_selling=products.orderBy(col("total_sales").desc()).limit(1)
top_selling.display()

product,total_sales
Laptop,210000.0


In [0]:
top_customer=customers.orderBy(col("total_order").desc()).limit(1)
top_customer.display()

order_id,total_order
103,4
